In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import NMF
import os

# Create output dirs if they don't exist
os.makedirs("results/nmf", exist_ok=True)

print("--- MATRIX FACTORIZATION LAB ---")
print("Student Roll No: 24BAD067 Manoj M\n")

# 1. Load the dataset
print("Loading MovieLens 100k dataset...")
cols = ['user_id', 'movie_id', 'rating', 'timestamp']
try:
    train_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u1.base', sep='\t', names=cols, encoding='latin-1')
    test_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u1.test', sep='\t', names=cols, encoding='latin-1')
except FileNotFoundError:
    print("Error: Could not find the 'ml-100k' dataset folder. Make sure to download and extract it.")
    exit(1)

# Load movie names for recommendations
item_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
try:
    items_df = pd.read_csv('/Users/manojmj/PycharmProjects/machinelearning subject/expno10/ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1', usecols=['movie_id', 'movie_title'])
    movie_dict = dict(zip(items_df.movie_id, items_df.movie_title))
except FileNotFoundError:
    print("Error: Could not find 'ml-100k/u.item'.")
    exit(1)

n_users = max(train_df.user_id.max(), test_df.user_id.max())
n_items = max(train_df.movie_id.max(), test_df.movie_id.max())

print(f"Dataset loaded: {n_users} users, {n_items} movies.")
print(f"Train set interactions: {len(train_df)}")
print(f"Test set interactions: {len(test_df)}\n")

# 2. Data Preprocessing: Create User-Item Interaction Matrix
R_df = train_df.pivot(index='user_id', columns='movie_id', values='rating')
# Fill missing movies to ensure matrix dimensions match max users/movies
R_df = R_df.reindex(index=range(1, n_users + 1), columns=range(1, n_items + 1))

# NMF needs strictly non-negative matrix, fillna works perfectly.
R = R_df.fillna(0).values
R_nmf_input = R.copy()

print("--- SCENARIO 2: NMF ---")
k_values_nmf = [5, 10, 20, 50]
rmse_list_nmf = []

best_k_nmf = 20
R_pred_best_nmf = None
best_H = None

print("Evaluating NMF for different k values...")
for k in k_values_nmf:
    model = NMF(n_components=k, init='random', random_state=42, max_iter=1000)
    W = model.fit_transform(R_nmf_input)
    H = model.components_

    R_pred = np.dot(W, H)

    if k == best_k_nmf:
        R_pred_best_nmf = R_pred
        best_H = H

    test_preds = []
    test_actuals = []
    for _, row in test_df.iterrows():
        u = int(row['user_id']) - 1
        i = int(row['movie_id']) - 1
        test_actuals.append(row['rating'])
        pred = max(1, min(5, R_pred[u, i]))
        test_preds.append(pred)

    rmse = np.sqrt(mean_squared_error(test_actuals, test_preds))
    rmse_list_nmf.append(rmse)
    print(f"  k={k:3d} -> RMSE: {rmse:.4f}")

# Plot Latent Features (H matrix from NMF for best K)
plt.figure(figsize=(10, 6))
sns.heatmap(best_H[:5, :50], cmap="magma") # first 5 components, 50 items
plt.title("NMF Latent Features (Item-Feature Matrix H) - Top 5 components")
plt.xlabel("Movie IDs")
plt.ylabel("Latent Feature Dimension")
plt.savefig("results/nmf/latent_features_H.png")
plt.close()

# Reconstruction comparison
plt.figure()
plt.scatter(R_nmf_input[:10, :100].flatten(), R_pred_best_nmf[:10, :100].flatten(), alpha=0.3)
plt.plot([-1, 6], [-1, 6], 'r--')
plt.xlim(-0.5, 5.5)
plt.ylim(-0.5, 5.5)
plt.title("NMF Reconstruction Comparison (Actual vs Predicted)")
plt.xlabel("Actual Ratings (0 means unrated)")
plt.ylabel("Predicted Ratings")
plt.savefig("results/nmf/reconstruction_comparison.png")
plt.close()

# Generate Top-N recommendations for a sample user
target_user = 1
u_idx = target_user - 1
user_seen_movies = train_df[train_df['user_id'] == target_user]['movie_id'].values

preds_nmf = R_pred_best_nmf[u_idx, :]
top_indices_nmf = preds_nmf.argsort()[::-1]

recommendations_nmf = []
count = 0
print(f"\nTop 10 NMF Recommendations for User {target_user}:")
for idx in top_indices_nmf:
    movie_id = idx + 1
    if movie_id not in user_seen_movies:
        title = movie_dict.get(movie_id, "Unknown Movie")
        score = preds_nmf[idx]
        print(f"  {count+1}. {title} (Predicted Rating: {score:.2f})")
        recommendations_nmf.append((title, score))
        count += 1
        if count == 10:
            break

plt.figure(figsize=(10, 6))
recom_titles_nmf = [r[0] for r in recommendations_nmf]
recom_scores_nmf = [r[1] for r in recommendations_nmf]
sns.barplot(x=recom_scores_nmf, y=recom_titles_nmf, hue=recom_titles_nmf, palette="Reds_r", legend=False)
plt.title(f"NMF: Top 10 Recommended Movies for User {target_user}")
plt.xlabel("Predicted Rating")
plt.savefig("results/nmf/recommendation_ranking.png", bbox_inches='tight')
plt.close()

print(f"\nFinal Chosen NMF (k={best_k_nmf}) RMSE: {rmse_list_nmf[2]:.4f}")

# Precision@K and Recall@K calculation
def precision_recall_at_k(predictions, threshold=3.5, k=10):
    user_est_true = {}
    for uid, iid, true_r, est in predictions:
        if uid not in user_est_true:
            user_est_true[uid] = []
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    print(f"\nMetrics at K={k} (rating threshold >={threshold})")
    print(f"  Precision@{k}: {sum(precisions.values()) / len(precisions):.4f}")
    print(f"  Recall@{k}:    {sum(recalls.values()) / len(recalls):.4f}")

preds_list_nmf = []
for _, row in test_df.iterrows():
    u = int(row['user_id']) - 1
    i = int(row['movie_id']) - 1
    actual = row['rating']
    est = R_pred_best_nmf[u, i]
    preds_list_nmf.append((u, i, actual, est))

precision_recall_at_k(preds_list_nmf, k=10)

print("\n--- ANALYSIS COMPLETED ---")


--- MATRIX FACTORIZATION LAB ---
Student Roll No: 24BAD067 Manoj M

Loading MovieLens 100k dataset...
Dataset loaded: 943 users, 1682 movies.
Train set interactions: 80000
Test set interactions: 20000

--- SCENARIO 2: NMF ---
Evaluating NMF for different k values...
  k=  5 -> RMSE: 2.6007
  k= 10 -> RMSE: 2.5646
  k= 20 -> RMSE: 2.5501
  k= 50 -> RMSE: 2.5810

Top 10 NMF Recommendations for User 1:
  1. Fargo (1996) (Predicted Rating: 3.51)
  2. Raiders of the Lost Ark (1981) (Predicted Rating: 2.74)
  3. Contact (1997) (Predicted Rating: 2.57)
  4. Groundhog Day (1993) (Predicted Rating: 2.54)
  5. Usual Suspects, The (1995) (Predicted Rating: 2.52)
  6. Indiana Jones and the Last Crusade (1989) (Predicted Rating: 2.49)
  7. Pulp Fiction (1994) (Predicted Rating: 2.47)
  8. This Is Spinal Tap (1984) (Predicted Rating: 2.37)
  9. Blues Brothers, The (1980) (Predicted Rating: 2.34)
  10. Star Trek: First Contact (1996) (Predicted Rating: 2.20)

Final Chosen NMF (k=20) RMSE: 2.5501

Met